# Day 12: Working with Dates & Timestamps

### Built-in functions for time manipulation in Spark

If you’ve worked with spreadsheets before, you’ve probably seen how messy dates can get. One cell says 01/09/2025, another says 2025-09-01, and a third is blank or has "N/A". Now imagine this at the scale of millions of rows across dozens of systems—finance, sales, customer activity logs, IoT events. Time becomes one of the trickiest parts of a data engineer’s life.

The good news is that Spark gives us a rich set of built-in functions to tame this complexity. Today, we’ll explore how to read, clean, and manipulate dates and timestamps in Spark. We’ll move step by step: from raw strings, to Spark’s internal date and timestamp types, to transformations like extracting months, calculating differences, and shifting time windows.

###  From Strings to Dates
Let’s return to our sales.csv, now with dates included:

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("day12-dates").getOrCreate()

df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/12_sales_csv.csv")
df.printSchema()

Notice how Spark read date as a string, not as a proper date. This is a common gotcha. Strings look like dates, but Spark can’t perform date math on them until we explicitly convert.

We use the to_date function:

In [0]:
from pyspark.sql.functions import to_date, col

df = df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
df.printSchema()

Now date shows as:

At this point, Spark understands it as an actual date type.

From Strings to Timestamps
What if we also had timestamps with hours, minutes, seconds?

region,store,revenue,transaction_time

North,StoreA,1000,2025-09-01 09:15:00

South,StoreB,1500,2025-09-02 14:30:45

We can convert this into Spark’s timestamp type:

In [0]:
from pyspark.sql.functions import to_timestamp
df =  spark.read.csv("/Volumes/thedataengineering_00/data/sampledata/12_sales_transaction_time.csv", header=True, inferSchema=True)
df = df.withColumn("transaction_time", to_timestamp(col("transaction_time"), "yyyy-MM-dd HH:mm:ss"))
df.printSchema()

This gives us a timestamp column, which is richer than date because it carries both the calendar date and the exact time of day.

### Extracting Parts of a Date or Time
Once we have proper date/timestamp types, we can start pulling out useful features. For example, business teams often want “month-to-date revenue” or “day of week analysis.”

In [0]:
from pyspark.sql.functions import year, month, dayofmonth, dayofweek, hour

df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/12_sales_csv.csv")

df = (
    df.withColumn("year", year(col("date")))
      .withColumn("month", month(col("date")))
      .withColumn("day", dayofmonth(col("date")))
      .withColumn("weekday", dayofweek(col("date")))
)

df.show()

This makes time queryable: you can group sales by month, check weekend vs weekday patterns, or slice by hour.

### Calculating Differences
One of the most common tasks in data engineering is calculating time differences. For example: “How many days between two orders?” or “How many minutes between login and logout?”

In [0]:
from pyspark.sql.functions import datediff, unix_timestamp, current_date

df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/12_sales_csv.csv")

# Days between current date and order date
df = df.withColumn("days_since_order", datediff(current_date(), col("date")))


display(df)

For finer granularity, Spark stores timestamps as seconds since epoch (unix_timestamp), so differences give you exact seconfting Dates

Sometimes you don’t want to just measure differences — you want to shift dates forward or backward. Maybe finance needs to project revenue one month ahead, or you need to backdate by a week.

In [0]:
from pyspark.sql.functions import add_months, date_add, date_sub

df = (
    df.withColumn("next_month", add_months(col("date"), 1))
      .withColumn("plus_7_days", date_add(col("date"), 7))
      .withColumn("minus_7_days", date_sub(col("date"), 7))
)

df.show()

### Formatting and Presentation
Once your data is processed, you may need to display dates in a specific format for reporting. Spark’s date_format helps here:

In [0]:
from pyspark.sql.functions import date_format

df = df.withColumn("pretty_date", date_format(col("date"), "dd-MMM-yyyy"))
df.show(truncate=False)

### Time Zones — The Silent Gotcha
Timestamps become tricky when time zones get involved. By default, Spark interprets timestamps using the system or session time zone. If your data comes from different regions, you may need to normalize everything to UTC.

In [0]:
from pyspark.sql.functions import from_utc_timestamp, to_utc_timestamp

# Convert stored UTC timestamp to IST (Asia/Kolkata)
df = df.withColumn("txn_time_ist", from_utc_timestamp(col("transaction_time"), "Asia/Kolkata"))

If you don’t handle this carefully, a midnight purchase in New York could show up as “next day” in London.

Why This Matters for Data Engineering
Dates and timestamps are not just “columns.” They are the backbone of most analysis:

A retailer’s sales dashboard depends on daily, weekly, and monthly aggregates.

A streaming pipeline uses event time to decide windows for aggregation.

A finance pipeline uses settlement dates and due dates for reconciliation.

If you mishandle dates — say, by leaving them as strings or ignoring time zones — you risk delivering reports that look correct but are subtly wrong. And nothing damages trust in a data pipeline faster than inconsistent time values.

# Wrapping Up
Today you learned how Spark treats time as a first-class citizen:

- Always convert strings into proper date or timestamp types.
- Extract parts like year, month, day, or hour for grouping.
- Use built-in functions like datediff, date_add, add_months to measure and manipulate.
- Format dates for presentation only at the end.
- Be mindful of time zones — they’re invisible until they bite you.

With these tools, you can build reliable, time-aware pipelines that finance, operations, and analytics teams can trust.

That’s Day 12. You now have a strong foundation for working with dates and timestamps in Spark.